# CC-WBT-AI — Input Pipeline

Workflow: set inputs → write to files → run engine → read outputs

In [1]:
import sys, os, json
 
_cwd = os.path.abspath('')
if os.path.exists(os.path.join(_cwd, 'backend')):
    ROOT = _cwd
elif os.path.exists(os.path.join(_cwd, '..', 'backend')):
    ROOT = os.path.dirname(_cwd)
else:
    raise FileNotFoundError("Cannot find the backend folder")
 
BACKEND   = os.path.join(ROOT, 'backend')
NOTEBOOKS = os.path.join(ROOT, 'notebooks')
sys.path.insert(0, BACKEND)
sys.path.insert(0, NOTEBOOKS)

from pipeline_helpers import PipelineIO
print('Setup OK')

Setup OK


## 1. Model and technologies

In [2]:
COUNTRY    = 'Mozambique'
MODEL      = 'CleanStep'
TECHS      = ['Electricity', 'LPG']
START_YEAR = 2023
END_YEAR   = 2034
YEARS      = list(range(START_YEAR, END_YEAR + 1))
 
pio = PipelineIO(backend=BACKEND, country=COUNTRY, model=MODEL, techs=TECHS, start_year=START_YEAR, end_year=END_YEAR)
pio.ensure_model_exists()

  Copied fuel-financial-inputs.xlsx from template
  Copied carbon-credits.xlsx from template
Files to recreate from templates:
  design-capital-CleanStep.xlsx: does not exist
  technoeconomic-inputs-CleanStep.xlsx: does not exist
  financial-statements-CleanStep.xlsx: does not exist
  capex-fuels-CleanStep.xlsx: does not exist
  design-capital-CleanStep.xlsx
  technoeconomic-inputs-CleanStep.xlsx
  financial-statements-CleanStep.xlsx
  capex-fuels-CleanStep.xlsx
Model CleanStep ready.


## 2. Country-level inputs

In [3]:
# config
TAX_RATE  = 28
INFLATION = 5
 
# fuel financial inputs
FUEL_FIN = {
    'Electricity': {
        'NTL':      [5]*12,
        'DAYS_REC': [30]*12,
        'DAYS_PAY': [30]*12,
    },
    'LPG': {
        'NTL':      [5]*12,
        'DAYS_REC': [30]*12,
        'DAYS_PAY': [30]*12,
    },
}
 
# carbon credits (fuel-financial-inputs)
CC_CO2_CERT  = [0]*12
CC_LIQUIDITY = [0]*12
CC_PRICE_TON = [0]*12
CC_NUM_YEARS = 0
 
print('Country-level inputs defined.')

Country-level inputs defined.


## 3. Model-level inputs

In [4]:
# capital structure (design-financial-structure)
CAPITAL_STRUCTURE = {
    'Electricity & E-Cooking': {
        'PCT_EQUITY':        20,
        'COST_OF_EQUITY':    16,
        'PCT_GRANTS':        50,
        'YEARS_REALISATION': 8,
        'COST_OF_DEBT':      8,
        'GRACE_PERIOD':      6,
        'AMORTIZATION':      25,
        'DEBT_INCREASE':     [100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    },
    'Electricity (Low access)': {
        'PCT_EQUITY':        25,
        'COST_OF_EQUITY':    12,
        'PCT_GRANTS':        55,
        'YEARS_REALISATION': 8,
        'COST_OF_DEBT':      8,
        'GRACE_PERIOD':      6,
        'AMORTIZATION':      25,
        'DEBT_INCREASE':     [0]*12,
    },
    'LPG': {
        'PCT_EQUITY':        50,
        'COST_OF_EQUITY':    12,
        'PCT_GRANTS':        20,
        'YEARS_REALISATION': 2,
        'COST_OF_DEBT':      8,
        'GRACE_PERIOD':      6,
        'AMORTIZATION':      25,
        'DEBT_INCREASE':     [0]*12,
    },
}
 
# technoeconomic-inputs - Electricity (GRID/OFF-GRID)
TECHNO_ELEC = {
    'Electricity & E-Cooking': {
        'GRID': {
            'Demand':             [1473.6, 1616.8, 1793.2, 1954.5, 3101.6, 2235.1, 2355.9, 2518.5, 1659.5, 2786.4, 2900, 3001],
            'CAPEX - Growth':     [257.3,  324.6,  321.4,  318.2, 315, 311.8, 382, 451.1, 446.6, 442.1, 437.7, 290.8],
            'CAPEX - Maintenance':[0]*12,
            'OPEX':               [48.2,   54.6,   62.2,   69.6, 77, 84.1, 91.1, 102.2, 112.7, 123, 133.1, 143],
            'D&A':                35,
        },
        'OFF-GRID': {
            'Demand':             [368.4, 379.3, 393.6, 400.3, 400.3, 394.4, 383.5, 376.3, 362.7, 344.4, 322.2, 296.8],
            'CAPEX - Growth':     [39.5,  50.3, 49.8, 49.3, 48.8, 48.4, 15.4, -17.1, -16.9, -16.8, -16.6, -1.7],
            'CAPEX - Maintenance':[0]*12,
            'OPEX':               [56.1,  64.7, 74.9, 84.8, 94.5, 104, 113.4, 111.6, 105.1, 98.8, 92.6, 86.4],
            'D&A':                25,
        },
    },
    'Electricity (Low access)': {
        'GRID': {
            'Demand':              [1443, 1496, 1584, 1666, 1742, 1811, 1875, 1934, 1991, 2044, 2092, 2136],
            'CAPEX - Growth':      [215,   248,   246,   243,   241,   238,   276,   313,   310,   307,   304,   209  ],
            'CAPEX - Maintenance': [0]*12,
            'OPEX':                [46,    48,    51,    54,    57,    59,    62,    71,    81,    91,    101,   111  ],
            'D&A': 35,
        },
        'OFF-GRID': {
            'Demand':              [368,  379,  394,  400,  400,  394,  384,  376,  363,  344,  322,  297],
            'CAPEX - Growth':      [40,   50,   50,   49,   49,   48,   15,   -17,  -17,  -17,  -17,  -2 ],
            'CAPEX - Maintenance': [0]*12,
            'OPEX':                [56,   65,   75,   85,   95,   104,  113,  112,  105,  99,   93,   86 ],
            'D&A': 25,
        },
    },
}
 
# technoeconomic-inputs — LPG (other fuels)
TECHNO_LPG = {
    'Demand':              [302,  327,  439,  539,  629,  710,  781,  970,  949,  928,  907,  885],
    'CAPEX - Growth':      [3,    5,    5,    5,    5,    5,    4,    3,    3,    3,    3,    3  ],
    'CAPEX - Maintenance': [0]*12,
    'OPEX':                [25,   36,   51,   66,   81,   96,   111,  115,  118,  121,  123,  126],
    'D&A': 15,
}
 
# tariffs & upstreams
TARIFFS = {
    'Electricity': [0.15, 0.16, 0.17, 0.17, 0.18, 0.19, 0.20, 0.21, 0.22, 0.23, 0.24, 0.26],
    'LPG':         [0.08, 0.08, 0.08, 0.09, 0.09, 0.10, 0.10, 0.11, 0.11, 0.12, 0.12, 0.13],
}
UPSTREAMS = {
    'Electricity': [0]*12,
    'LPG':         [0]*12,
}
 
# CO2 emitted (carbon-credits)
CO2_BASELINE = [21.61, 20.602, 19.838, 19.074, 18.311, 17.547, 16.783, 16.436, 16.367, 16.298, 16.229, 16.160]
CO2_EMITTED = [21.611, 19.405, 17.428, 15.452, 13.475, 11.498, 9.521, 9.105, 9.190, 9.275, 9.360, 9.446]
 
print('Model-level inputs defined.')

Model-level inputs defined.


## 4. Write inputs

In [5]:
print('=== COUNTRY-LEVEL ===')
pio.write_config_scalar('TAX_RATES',  TAX_RATE)
pio.write_config_scalar('INFLATIONS', INFLATION)
 
for fuel, vals in FUEL_FIN.items():
    pio.write_timeseries_list(pio.fuel_fin_path, fuel, 'Expected non-technical losses',        vals['NTL'])
    pio.write_timeseries_list(pio.fuel_fin_path, fuel, 'Trade receivables - Days of revenues', vals['DAYS_REC'])
    pio.write_timeseries_list(pio.fuel_fin_path, fuel, 'Trade payables - Days of OPEX costs',  vals['DAYS_PAY'])
 
pio.write_timeseries_list      (pio.fuel_fin_path, 'Carbon Credits', 'CO2 certificate (%) - from the total CO2 avoided', CC_CO2_CERT)
pio.write_timeseries_list      (pio.fuel_fin_path, 'Carbon Credits', 'Liquidity',     CC_LIQUIDITY)
pio.write_timeseries_list      (pio.fuel_fin_path, 'Carbon Credits', 'Price per TON', CC_PRICE_TON)
pio.write_timeseries_first_year(pio.fuel_fin_path, 'Carbon Credits', 'Number of years that you could sell those carbon credits', CC_NUM_YEARS)
 
print('=== MODEL-LEVEL ===')
for sheet, cs in CAPITAL_STRUCTURE.items():
    pio.write_excel_scalar(pio.design_cap_path, sheet, '1. Equity',           cs['PCT_EQUITY'])
    pio.write_excel_scalar(pio.design_cap_path, sheet, 'Cost of Equity',      cs['COST_OF_EQUITY'])
    pio.write_excel_scalar(pio.design_cap_path, sheet, '2. Grants',           cs['PCT_GRANTS'])
    pio.write_excel_scalar(pio.design_cap_path, sheet, 'Years realisation',   cs['YEARS_REALISATION'])
    pio.write_excel_scalar(pio.design_cap_path, sheet, 'Cost of Debt',        cs['COST_OF_DEBT'])
    pio.write_excel_scalar(pio.design_cap_path, sheet, 'Grace period',        cs['GRACE_PERIOD'])
    pio.write_excel_scalar(pio.design_cap_path, sheet, 'Amortization period', cs['AMORTIZATION'])
    pio.write_design_debt_list(sheet, cs['DEBT_INCREASE'])
 
for sheet, systems in TECHNO_ELEC.items():
    for system, params in systems.items():
        for param, values in params.items():
            if param == 'D&A': pio.write_techno_da(sheet, system, values)
            else:               pio.write_techno_list(sheet, system, param, values)
 
for param, values in TECHNO_LPG.items():
    if param == 'D&A': pio.write_techno_nosystem_da('LPG', values)
    else:               pio.write_techno_nosystem_list('LPG', param, values)
 
for tech, vals in TARIFFS.items():
    pio.write_config_timeseries('TARIFFS',   tech, YEARS, vals)
for tech, vals in UPSTREAMS.items():
    pio.write_config_timeseries('UPSTREAMS', tech, YEARS, vals)
 
pio.write_carbon_row('CO2 emited - Baseline scenario', CO2_BASELINE)
pio.write_carbon_co2_emitted(CO2_EMITTED, YEARS)
print('\nAll inputs written.')

=== COUNTRY-LEVEL ===
  config.json -> TAX_RATES = 28
  config.json -> INFLATIONS = 5
  fuel-financial-inputs.xlsx / Electricity -> Expected non-technical losses
  fuel-financial-inputs.xlsx / Electricity -> Trade receivables - Days of revenues
  fuel-financial-inputs.xlsx / Electricity -> Trade payables - Days of OPEX costs
  fuel-financial-inputs.xlsx / LPG -> Expected non-technical losses
  fuel-financial-inputs.xlsx / LPG -> Trade receivables - Days of revenues
  fuel-financial-inputs.xlsx / LPG -> Trade payables - Days of OPEX costs
  fuel-financial-inputs.xlsx / Carbon Credits -> CO2 certificate (%) - from the total CO2 avoided
  fuel-financial-inputs.xlsx / Carbon Credits -> Liquidity
  fuel-financial-inputs.xlsx / Carbon Credits -> Price per TON
  fuel-financial-inputs.xlsx / Carbon Credits -> Number of years that you could sell those carbon credits (first year)
=== MODEL-LEVEL ===
  design-capital-CleanStep.xlsx / Electricity & E-Cooking -> 1. Equity = 20
  design-capital-Clea

## 5. Run engine

In [6]:
print('Running engine...')
pio.run_engine()
print('Done.')

Running engine...
Done.


## 6. Outputs

In [7]:
SHEET = 'Electricity & E-Cooking'

with open(pio.config_path) as f:
    _cfg = json.load(f)
YEARS = list(range(_cfg['COUNTRY_YEAR_RANGES'][COUNTRY]['start'],
                   _cfg['COUNTRY_YEAR_RANGES'][COUNTRY]['end'] + 1))
 
out   = pio.read_outputs(SHEET)
col_w = 10
header = f"{'Row':<35}" + ''.join(f"{y:>{col_w}}" for y in YEARS)
print(header)
print('-' * len(header))
for key, values in out.items():
    print(f"{key:<35}" + ''.join(f"{round(v,1):>{col_w}}" for v in values))
 

Row                                      2023      2024      2025      2026      2027      2028      2029      2030      2031      2032      2033      2034
-----------------------------------------------------------------------------------------------------------------------------------------------------------
Long term subsidies                         0         0         0         0         0         0         0         0         0         0         0         0
Operating Cash Flow                     -14.1      -2.3      -2.8      -0.9     -17.5      12.1      -2.6      -4.2      13.7     -22.3      -4.1      -6.6
Financial Expense                           0         0         0         0         0         0         0         0         0         0         0         0
Debt repayment                              0         0         0         0         0         0         0         0         0         0         0         0
Equity - EoP                                0         0         

## 7. Summary metrics

In [8]:
lts      = out.get('Long term subsidies',  [0]*len(YEARS))
equity   = out.get('Equity - EoP',         [0]*len(YEARS))
debt_eop = out.get('DEBT:EoP',             [0]*len(YEARS))
grants   = out.get('GRANTS:- Realisation', [0]*len(YEARS))
 
wacc         = PCT_EQUITY/100 * COST_OF_EQUITY + (100 - PCT_EQUITY - PCT_GRANTS)/100 * COST_OF_DEBT
total_lts    = round(sum(lts), 1)
total_grants = round(sum(abs(v) for v in grants), 1)
peak_equity  = round(max(equity), 1)
peak_debt    = round(max(abs(v) for v in debt_eop), 1)
 
print(f"WACC:               {wacc:.2f}%")
print(f"Total LTS:          {total_lts} M$")
print(f"Total grants:       {total_grants} M$")
print(f"Peak equity (EoP):  {peak_equity} M$")
print(f"Peak debt (EoP):    {peak_debt} M$")

NameError: name 'PCT_EQUITY' is not defined

In [ ]:
_grid   = TECHNO_ELEC.get('Electricity & E-Cooking', {}).get('GRID', {})
_demand = _grid.get('Demand', [])
_capex  = _grid.get('CAPEX - Growth', [])
_opex   = _grid.get('OPEX', [])
 
K1 = [round(_capex[i] / _demand[i], 4) if _demand[i] else 0 for i in range(len(YEARS))]
K2 = [round(_capex[i] / _opex[i],   4) if _opex[i]   else 0 for i in range(len(YEARS))]
print(f"K1 (CAPEX/Demand): {K1}")
print(f"K2 (CAPEX/OPEX):   {K2}")

K1 (CAPEX/Demand): [0.1746, 0.2008, 0.1792, 0.1628, 0.1016, 0.1395, 0.1621, 0.1791, 0.2691, 0.1587, 0.1509, 0.0969]
K2 (CAPEX/OPEX):   [5.3382, 5.9451, 5.1672, 4.5718, 4.0909, 3.7075, 4.1932, 4.4139, 3.9627, 3.5943, 3.2885, 2.0336]
